In [5]:
# %%
import time
import numpy as np
import os
# from tools import *

from IPython import get_ipython

# Ensure the Qt event loop is integrated with Jupyter
# get_ipython().run_line_magic('gui', 'qt')
from PySide2.QtCore import QTimer, QTime, Signal
%gui qt

In [101]:
# %%
import time
import numpy as np
import json
import schedule
import os
from toptica.lasersdk.dlcpro.v2_0_3 import DLCpro,LaserHead,  NetworkConnection, DeviceNotFoundError

#from 0 to 19650 MHz (max val defined by config) laser offset 
def go_to_ple_target(target):
    ple_gui._mw.ple_widget.target_point.setValue(target)
    ple_gui._mw.ple_widget.target_point.sigPositionChangeFinished.emit(target)
    #laser_scanner_logic.set_target_position({"a": target})
    time.sleep(0.5)

#save confocal map. It will be saved in the folder defined in the app
def save_scan(name):
    scanner_gui.save_path_widget.saveTagLineEdit.setText(name)
    scanner_gui.scan_2d_dockwidgets[('x', 'y')].scan_widget.save_scan_button.clicked.emit() #saving
    dir_ = scanning_data_logic.module_default_data_dir
    return name, dir_

#laser offset with the topica:
def set_laser_offset(v):
    with DLCpro(NetworkConnection(dl_pro.tcp_address)) as dlc:
        dlc._laser1.dl.pc.voltage_set.set(v)

def get_laser_offset():
    with DLCpro(NetworkConnection(dl_pro.tcp_address)) as dlc:
        v0 = dlc._laser1.dl.pc.voltage_set.get()
    return v0
#start scanning with toptica
def enable_laser_scanning(enable):
    with DLCpro(NetworkConnection(dl_pro.tcp_address)) as dlc:
        dlc._laser1.scan.enabled.set(enable)


def blue_quick_repump(dt = 0.01):
    # pulsestreamer._seq.setDigital(1, [(1000, 1)])
    pulsestreamer._seq.setDigital(2, [(1000, 1)])
    time.sleep(dt)
    pulsestreamer._seq.setDigital(2, [(1000, 0)])
    pulsestreamer.pulser_on()

#add the trigger for ple checking to be able to process out it in timetagger dump
def PLE_check_trigger(enable=True):
    # pulsestreamer._seq.setDigital(1, [(1000, 1)])
    pulsestreamer._seq.setDigital(5, [(100000000000, 0), (10, int(enable))])
    
    pulsestreamer.pulser_on()

def save_ple(tag, poi_name=None, folder_name = None):
        if folder_name:
            ple_gui._save_folderpath = folder_name
        ple_gui.save_path_widget.saveTagLineEdit.setText(
            f"{poi_name}_{tag}"
            )
        ple_gui._mw.actionSave.triggered.emit()

def do_ple_scan(lines = 1, in_range = None, frequency=None, resolution=None):
    """
    fine_scan_range = (
            self.ple_gui.fit_result[1].best_values['center'] - self.ple_gui.fit_result[1].best_values['sigma'] * 3,
            self.ple_gui.fit_result[1].best_values['center'] + self.ple_gui.fit_result[1].best_values['sigma']  * 3
        )
    """

    #laser_scanner_logic.scan_ranges["a"]
    if in_range is None:
        ple_gui._mw.actionFull_range.triggered.emit()
    else:
        ple_gui.sigScanSettingsChanged.emit(
            {
            'range': {ple_gui.scan_axis: in_range}
            }
        )
    ple_gui._mw.number_of_repeats_SpinBox.setValue(lines)
    ple_gui._mw.number_of_repeats_SpinBox.editingFinished.emit()
    time.sleep(0.5)
    ple_gui._mw.actionToggle_scan.setChecked(True)
    ple_gui.toggle_scan()
    while laser_scanner_logic.module_state()=='locked':
            time.sleep(1)
    time.sleep(1)
    ple_gui._fit_dockwidget.fit_widget.sigDoFit.emit("Lorentzian")
    time.sleep(1)
    # self.ple_gui._accumulated_data.mean(axis=0)
    print(f"Rsquared {ple_gui.fit_result[1].rsquared}")
    # self.ple_gui.fit_result[1].params["center"].value
    return ple_gui.fit_result[1]


def green_laser(turn_on = True):
    pulsestreamer._seq = pulsestreamer.pulse_streamer.createSequence()
    # pulse_pattern_cw_450 = [(_hist_record_length, 0), (10, 1)]
    pulsestreamer._seq.setDigital(1, [(1000, int(0))])
    pulsestreamer._seq.setDigital(3, [(1000, int(turn_on))])
    
    # timetagger.sigToggleHist.emit({'hist': (_hist_bin_width, _hist_record_length, int(hist_channel), False)})
    pulsestreamer.pulser_on()
    time.sleep(0.1)

def blue_laser_repump(enable=True, res = True, on_t = 1e5, off_t = 1e9):
    pulsestreamer._seq = pulsestreamer.pulse_streamer.createSequence()
    pulse_pattern_cw_450 = [(int(off_t), 0), (int(on_t), 1)]
    pulsestreamer._seq.setDigital(1, [(1000, int(res))])
    pulsestreamer._seq.setDigital(2, pulse_pattern_cw_450)
    pulsestreamer._seq.setDigital(3, [(1000, int(False))])
    
    
    # timetagger.sigToggleHist.emit({'hist': (_hist_bin_width, _hist_record_length, int(hist_channel), False)})
    pulsestreamer.pulser_on()
    time.sleep(0.1)

#PLE mode:

def measurement_mode(mode):
    if mode == "PLE":
        # red on, blue on, green off
        if switchlogic.get_state("Mirror") != 'On':
            switchlogic.set_state(switch = "Mirror", state = "On")
        time.sleep(1)
        if switchlogic.get_state("Mirror") == 'On':
            if switchlogic.get_state("Shutter") != 'Off':
                switchlogic.set_state(switch = "Shutter", state = "Off")
            time.sleep(0.5)
            if switchlogic.get_state("ResonantBF") != 'On':
                switchlogic.set_state(switch = "ResonantBF", state = "On")
            # if switchlogic.get_state("GreenBF") != 'Off':
            #     switchlogic.set_state(switch = "GreenBF", state = "Off")
            # if switchlogic.get_state("GreenAtto3") != 'Off':
            #     switchlogic.set_state(switch = "GreenAtto3", state = "Off")
            # if switchlogic.get_state("BlueBF") != 'On':
            #     switchlogic.set_state(switch = "BlueBF", state = "On")
            # if switchlogic.get_state("BlueAtto3") != 'On':
            #     switchlogic.set_state(switch = "BlueAtto3", state = "On")
            ibeam_smart_remote.power = 50
            powercontroller_logic._current_motor = 2
            powercontroller_logic.motor_position = 45 # green dim
            time.sleep(0.5)
        else:
            print("Mirror not in, PLE would burn APDs")
            raise BaseException
        # ibeam_remote.power = 0.01e3
        # blue_laser_repump(enable=True)

    elif mode == "Off-res":
        # green_laser(True)
        if switchlogic.get_state("ResonantBF") != 'Off':
                switchlogic.set_state(switch = "ResonantBF", state = "Off")
        if switchlogic.get_state("Shutter") != 'On':
            switchlogic.set_state(switch = "Shutter", state = "On")
        time.sleep(1)
        if switchlogic.get_state("Mirror") != 'Off':
            switchlogic.set_state(switch = "Mirror", state = "Off")
        # if switchlogic.get_state("GreenBF") != 'On':
        #     switchlogic.set_state(switch = "Green", state = "On")
        # if switchlogic.get_state("GreenAtto3") != 'On':
        #     switchlogic.set_state(switch = "GreenAtto3", state = "On")
        # if switchlogic.get_state("BlueBF") != 'Off':
        #         switchlogic.set_state(switch = "BlueBF", state = "Off")
        # if switchlogic.get_state("BlueAtto3") != 'Off':
        #     switchlogic.set_state(switch = "BlueAtto3", state = "Off")
        ibeam_smart_remote.power = 30e3
        powercontroller_logic._current_motor = 2
        powercontroller_logic.motor_position = 205 # MAX green
    else:
        print("No mode by this name")
def set_green_power(cryo, power):
    power_max = 40000
    power_min = 0
    if cryo == 'bf':
        min_position = 50
        max_position = 210
        # Ensure power is within bounds
        power = max(power_min, min(power, power_max))
        # Map power to the motor position range
        motor_position = min_position + (max_position - min_position) * (power - power_min) / (power_max - power_min)
        powercontroller_logic._current_motor = 2
        powercontroller_logic.motor_position = motor_position
        
    else:
        ibeam_smart_remote.power = power
# ws_wavemeter.start_acquisition()

In [6]:
measurement_mode(mode = 'PLE')

In [50]:
measurement_mode(mode = 'Off-res')

In [5]:
# switchlogic.set_state(switch = "Mirror", state = "Off")

In [41]:
switchlogic.set_state(switch = "Shutter", state = "Off")

In [34]:
ibeam_smart_remote.power = 20000


In [3]:
powercontroller_logic._current_motor = 2
powercontroller_logic.motor_position = 175 # MAX green

In [97]:
powercontroller_logic._current_motor = 2
powercontroller_logic.motor_position = 45 # green dim

In [32]:
powercontroller_logic._current_motor = 0
powercontroller_logic.motor_position = 30 # parallel pol

In [14]:
powercontroller_logic._current_motor = 0
powercontroller_logic.motor_position = 7 # perpendicular pol

In [35]:
cts_t = []
def check_counts():
    #%%
    ch1_cts = timetaggerlogic.counter.getDataNormalized()[0, :]
    ch2_cts = timetaggerlogic.counter.getDataNormalized()[1, :]

    # %%
    tot_cts = ch1_cts.mean() + ch2_cts.mean()
    # %%
    # cts_t.append(tot_cts / 1e3)
    # if cts_t <= 6500:

In [33]:
timer = QTimer()
timer.timeout.connect(check_counts)
timer.start(1000)  # 1 second interval

In [29]:
cts_t

[8.302200000000001, 8.3115, 8.312700000000001, 8.313799999999999, 8.2871, 8.29, 8.2641, 8.2642, 8.2789, 8.265799999999999, 8.238700000000001, 8.2607, 8.2406, 8.2499]

In [35]:
timer.stop()

In [62]:
ibeam_smart_remote.power = 10000

In [ ]:
ibeam_smart_remote.power = 50
powercontroller_logic._current_motor = 2
powercontroller_logic.motor_position = 240

In [39]:
powercontroller_logic._current_motor = 2
powercontroller_logic.motor_position = 220

In [91]:
ibeam_smart_remote.power = 5000

In [118]:
powercontroller_logic._current_motor = 0
powercontroller_logic.motor_position = 5 # perpendicular pol

In [119]:
powercontroller_logic._current_motor = 0
powercontroller_logic.motor_position = 22 # parallel pol

## Power dependent g2

In [122]:
poi_manager_logic_remote._optimizelogic()._last_fit_results.rsquared

0.9504635710435794

## Power dependent g2

In [210]:
from functools import partial

refocused_cts = None
class AutoMeasurements:
    def __init__(self, 
                 timetaggerlogic, 
                 timetagger, 
                 poi_manager_logic_remote, ) -> None:
        pass

    def check_counts(self):
        ch2_cts = timetaggerlogic.trace_data[2] #timetaggerlogic.counter.getDataNormalized()[0, :]
        ch3_cts = timetaggerlogic.trace_data[3]  #timetaggerlogic.counter.getDataNormalized()[1, :]

        # %%
        tot_cts = ch2_cts.mean() + ch3_cts.mean()
        refocused_cts = tot_cts if refocused_cts is None else refocused_cts

        if tot_cts <= refocused_cts * 0.75:
            refocus()
        timer.start(200 * 1000)


    def refocus(self):
        ch2_cts = timetaggerlogic.trace_data[2] #timetaggerlogic.counter.getDataNormalized()[0, :]
        ch3_cts = timetaggerlogic.trace_data[3]  #timetaggerlogic.counter.getDataNormalized()[1, :]

        # %%
        tot_cts = ch2_cts.mean() + ch3_cts.mean()
        
        save_tagger_plots(folder_g2_save, str(power))

        cts_refocus.append(tot_cts / 1e3)
        powercontroller_logic._current_motor = 0
        powercontroller_logic.motor_position = 5 # perpendicular pol
        time.sleep(3)

        poi_manager_logic_remote._optimizelogic().start_optimize()
        # poi_manager_logic._optimizelogic().start_optimize()
        while poi_manager_logic_remote._optimizelogic().module_state()=='locked':
            time.sleep(1) # wait for a long time to 
        time.sleep(20) # wait for a long time to avoid conflicts with the countrate checker

        powercontroller_logic._current_motor = 0
        powercontroller_logic.motor_position = 22 # parallel pol
        time.sleep(3)

        #%%
        ch2_cts = timetaggerlogic.trace_data[2] #timetaggerlogic.counter.getDataNormalized()[0, :]
        ch3_cts = timetaggerlogic.trace_data[3]  #timetaggerlogic.counter.getDataNormalized()[1, :]

        # %%
        refocused_cts = ch2_cts.mean() + ch3_cts.mean()

    def toggle_tagger_counter_plot(self, state):
        # start/ stop the counter measurement
        timetagger._mw.toggleCounterPushButton.setChecked(state)
        timetagger._mw.toggleCounterPushButton.toggled.emit(state)
        
    def toggle_tagger_corr_plot(self, state):
        # start/ stop the counter measurement
        timetagger._mw.toggleCorrPushButton.setChecked(state)
        timetagger._mw.toggleCorrPushButton.toggled.emit(state)

    def save_tagger_plots(self, folder, tag):
        timetagger._save_folderpath = folder
        timetagger._mw.currPathLabel.setText(folder)
        timetagger._mw.saveTagLineEdit.setText(tag)     
        timetagger._mw.counter_checkBox.setChecked(True)
        timetagger._save_data_clicked()
        timetagger._mw.corr_checkBox.setChecked(True)
        timetagger._save_data_clicked()

    def _g2_power_dependent(self, powers):
        if powers is None:
            return
        else:
            timer.start(10000)  # 1 second interval
            refocus_timer.start(25 * 60000)  # each 20 mins second interval
        if len(powers) < 1:
            integration_timer.stop()
            refocus_timer.stop()
            timer.stop()
            stop_dump()
        power = powers.pop()

        toggle_tagger_counter_plot(False)
        toggle_tagger_corr_plot(False)
        stop_dump()
        measurement_mode(mode = 'Off-res')
        
        refocus()

        set_green_power(current_cryo, power)
        set_green_power(non_active_cryo, 0)
        
        # start the measurement:
        toggle_tagger_counter_plot(True)
        toggle_tagger_corr_plot(True)
        start_dump(folder_g2_save, str(power))

        integration_timer.start(integrate_for_mins * 60e3)  # integrate in minutes

    def start_dump(self, folder, tag):
        pth = os.path.join(folder, str(tag))
        os.makedirs(pth, exist_ok = True)
        timetagger._mw.saveDumpTagLineEdit.setText(tag)
        timetagger._save_dump_folderpath = pth
        timetagger._mw.currDumpPathLabel.setText(timetagger._save_dump_folderpath)
        
        timetagger._mw.dump_checkBox.setChecked(True)
        timetagger._dump_toggled()

    def stop_dump(self):
        timetagger._mw.dump_checkBox.setChecked(False)
        timetagger._dump_toggled()

    
def measurement_mode(mode):
    if mode == "PLE":
        # red on, blue on, green off
        if switchlogic.get_state("Mirror") != 'On':
            switchlogic.set_state(switch = "Mirror", state = "On")
        time.sleep(1)
        if switchlogic.get_state("Mirror") == 'On':
            if switchlogic.get_state("Shutter") != 'Off':
                switchlogic.set_state(switch = "Shutter", state = "Off")
            time.sleep(0.5)
            if switchlogic.get_state("ResonantBF") != 'On':
                switchlogic.set_state(switch = "ResonantBF", state = "On")
            # if switchlogic.get_state("GreenBF") != 'Off':
            #     switchlogic.set_state(switch = "GreenBF", state = "Off")
            # if switchlogic.get_state("GreenAtto3") != 'Off':
            #     switchlogic.set_state(switch = "GreenAtto3", state = "Off")
            # if switchlogic.get_state("BlueBF") != 'On':
            #     switchlogic.set_state(switch = "BlueBF", state = "On")
            # if switchlogic.get_state("BlueAtto3") != 'On':
            #     switchlogic.set_state(switch = "BlueAtto3", state = "On")
            ibeam_smart_remote.power = 50
            powercontroller_logic._current_motor = 2
            powercontroller_logic.motor_position = 45 # green dim
            time.sleep(0.5)
        else:
            print("Mirror not in, PLE would burn APDs")
            raise BaseException
        # ibeam_remote.power = 0.01e3
        # blue_laser_repump(enable=True)

    elif mode == "Off-res":
        # green_laser(True)
        if switchlogic.get_state("ResonantBF") != 'Off':
                switchlogic.set_state(switch = "ResonantBF", state = "Off")
        if switchlogic.get_state("Shutter") != 'On':
            switchlogic.set_state(switch = "Shutter", state = "On")
        time.sleep(1)
        if switchlogic.get_state("Mirror") != 'Off':
            switchlogic.set_state(switch = "Mirror", state = "Off")
        # if switchlogic.get_state("GreenBF") != 'On':
        #     switchlogic.set_state(switch = "Green", state = "On")
        # if switchlogic.get_state("GreenAtto3") != 'On':
        #     switchlogic.set_state(switch = "GreenAtto3", state = "On")
        # if switchlogic.get_state("BlueBF") != 'Off':
        #         switchlogic.set_state(switch = "BlueBF", state = "Off")
        # if switchlogic.get_state("BlueAtto3") != 'Off':
        #     switchlogic.set_state(switch = "BlueAtto3", state = "Off")
        ibeam_smart_remote.power = 30e3
        powercontroller_logic._current_motor = 2
        powercontroller_logic.motor_position = 205 # MAX green
    else:
        print("No mode by this name")
def set_green_power(cryo, power):
    power_max = 40000
    power_min = 0
    if cryo == 'bf':
        min_position = 50
        max_position = 210
        # Ensure power is within bounds
        power = max(power_min, min(power, power_max))
        # Map power to the motor position range
        motor_position = min_position + (max_position - min_position) * (power - power_min) / (power_max - power_min)
        powercontroller_logic._current_motor = 2
        powercontroller_logic.motor_position = motor_position
        
    else:
        ibeam_smart_remote.power = power
# ws_wavemeter.start_acquisition()

In [207]:
powers = [10e3, 20e3, 25e3, 30e3, 40e3, 50e3]
# powers = [100, 125, 150, 175] #tentative

folder_g2_save = r'Z:\Vlad\SnV\TPI\Electrodes_e4\F1\atto3_D1\test_auto_power2'
current_cryo = 'atto3'
non_active_cryo = 'bf'

cts_refocus = []
cts_t = []

g2_power_dependent = partial(_g2_power_dependent, powers)

timer = QTimer()
timer.timeout.connect(check_counts) # rough check on drift to ensure it doesnt fly away
timer.setSingleShot(True)


refocus_timer = QTimer()
refocus_timer.timeout.connect(check_counts)
# refocus_timer.setSingleShot(True)
refocus_timer.start(25 * 60000)  # each 20 mins second interval

integration_timer = QTimer()
integration_timer.timeout.connect(g2_power_dependent) # rough check on drift to ensure it doesnt fly away
integration_timer.setSingleShot(True)

integrate_for_mins = 5
integration_timer.start(1000) #integrate_for_mins * 60e3)  # integrate in minutes

In [159]:
start_dump(folder_g2_save, str(5))

In [169]:
set_green_power(non_active_cryo, 0)


In [174]:
set_green_power(current_cryo, 50e3)

In [ ]:
poi_manager_logic_remote._optimizelogic().start_optimize()

0

In [ ]:
measurement_mode( mode = 'PLE')

In [ ]:
measurement_mode(mode = 'Off-res')

In [185]:
timetagger._mw.toggleCorrPushButton.isChecked()

False

In [211]:
refocus_timer.stop()
timer.stop()
integration_timer.stop()

## Voltage dependent g2 HOM

In [10]:
ao_electrodes_remote._current_channel = 'ao3'

In [154]:
ao_electrodes_remote.setpoint = -1.13

In [92]:
ibeam_smart_remote.power = 50

In [65]:
folder_g2_save = r'Z:\Vlad\SnV\TPI\Electrodes_e4\F1\atto3_D1\test_auto1'
current_cryo = 'atto3'
def V_dependent_hom(voltages = None):
    if voltages is None: return
    if len(voltages) < 1:
        integration_timer.stop()
        refocus_timer.stop()
        timer.stop()
        stop_dump()
    voltage = voltages.pop()
    measurement_mode(mode = 'Off-res')
    ao_electrodes_remote.setpoint = voltage
    # Make sure the dumping is off
    stop_dump()
    refocus()
    # start the measurement:
    start_dump(folder_g2_save, str(voltage))

In [148]:
measurement_mode(mode='PLE')

In [155]:
measurement_mode(mode='Off-res')

In [156]:
set_green_power('bf', 0)

In [149]:
set_green_power('atto3', 10000)

In [ ]:
poi_manager_logic_remote._optimizelogic().start_optimize()

0

In [ ]:
measurement_mode(mode = 'Off-res')